# CADOC 3040 — Geração de Arquivo XML Regulatório

Este notebook demonstra o pipeline de geração do arquivo **CADOC 3040**, relatório regulatório exigido pelo Banco Central do Brasil (BACEN) para instituições financeiras.

## O que este código faz
1. **Extrai dados** de um data warehouse (BigQuery) via SQL com CTEs
2. **Transforma** os dados com pandas
3. **Gera um XML** no formato exigido pelo BACEN

> ⚠️ **Nota:** Os dados neste notebook são **fictícios** e foram gerados apenas para fins de demonstração.

## 1. Dependências

In [ ]:
# Para rodar com BigQuery real, instale:
# pip install google-cloud-bigquery pandas db-dtypes

import pandas as pd
import math
from decimal import Decimal, ROUND_HALF_UP
from xml.etree.ElementTree import Element, SubElement
from io import StringIO

## 2. Query SQL — Extração do Data Warehouse

A query abaixo usa **CTEs (Common Table Expressions)** para organizar a extração em camadas:
- `cliente_conta`: vínculo entre clientes e produtos
- `cadastro3040`: dados cadastrais dos clientes 
- `total_cli`: contagem de clientes únicos por período
- `vencimentos3040`: distribuição de vencimentos por vértice de prazo
- `operacao3040`: dados das operações de crédito

Para execução real com BigQuery, substitua o bloco de dados fictícios abaixo pela autenticação via service account e execução da query.

In [ ]:
# Referência da query SQL utilizada em produção
# (executada via google.cloud.bigquery em ambiente real)

QUERY_REFERENCIA = """
WITH
  cliente_conta AS (
    SELECT
      person_id,
      a.id AS account_id,
      a.product_id,
      CASE a.product_id
        WHEN 2 THEN 'C1'
        WHEN 4 THEN 'C1'
        ELSE 'C5'
      END AS CartProvMin
    FROM dw.account AS a
  ),

  cadastro3040 AS (
    SELECT
      person_id,
      processing_date,
      authorization AS Autorzc,
      FORMAT_DATE('%Y-%m', DATE(processing_date)) AS DtBase,
      CASE person_type
        WHEN "1" THEN LEFT(national_registry, 11)  -- CPF
        ELSE LEFT(national_registry, 8)             -- CNPJ (raiz)
      END AS Cd,
      ROUND(annual_billing, 2) AS FatAnual,
      relationship_start_date AS IniRelactCli,
      customer_size AS PorteCli,
      person_type AS Tp
    FROM dw.bacen_customer
    INNER JOIN dw.people AS p ON person_id = p.id
  ),

  total_cli AS (
    SELECT DtBase, COUNT(DISTINCT Cd) AS TotalCli
    FROM cadastro3040
    GROUP BY DtBase
  ),

  vencimentos3040 AS (
    SELECT
      d.processing_date AS Dtbase,
      d.account_id,
      d.modality,
      ROUND(d.due_distribution_amount_1,  2) AS v110,
      ROUND(d.due_distribution_amount_2,  2) AS v120,
      ROUND(d.due_distribution_amount_14, 2) AS v220,
      ROUND(d.due_distribution_amount_16, 2) AS v240
      -- demais vertices omitidos neste exemplo
    FROM dw.bacen_due_date AS d
  ),

  operacao3040 AS (
    SELECT
      a.person_id,
      a.account_id,
      FORMAT_DATE('%Y-%m', DATE(processing_date)) AS DtBase,
      postal_code AS Cep,
      contract AS Contrt,
      modality AS Mod,
      effective_rate AS TaxEft,
      contract_amount AS VlrContr,
      delayed_days AS DiaAtraso,
      processing_date
    FROM dw.bacen_operation AS o
    INNER JOIN cliente_conta AS a ON o.account_id = a.account_id
    INNER JOIN dw.people AS p ON a.person_id = p.id
  )

SELECT
  c.DtBase, c.Autorzc, c.Cd, c.FatAnual, c.IniRelactCli,
  c.PorteCli, c.Tp,
  o.Cep AS CEP, o.Contrt, o.Mod, o.TaxEft, o.VlrContr, o.DiaAtraso,
  v.v110, v.v120, v.v220, v.v240,
  a.product_id, a.CartProvMin,
  tc.TotalCli
FROM operacao3040 AS o
INNER JOIN cadastro3040 AS c ON o.person_id = c.person_id
  AND c.processing_date = o.processing_date
LEFT JOIN vencimentos3040 AS v ON o.account_id = v.account_id
  AND o.Mod = v.modality AND o.processing_date = v.Dtbase
LEFT JOIN total_cli AS tc ON tc.DtBase = c.DtBase
LEFT JOIN cliente_conta AS a ON a.person_id = o.person_id
WHERE c.DtBase = '2025-01'  -- <-- alterar mensalmente
ORDER BY o.processing_date DESC;
"""

print("Query de referência carregada. Em produção, execute via bigquery.Client().")

## 3. Dados Fictícios para Demonstração

Em substituição ao BigQuery, carregamos um DataFrame com dados gerados artificialmente que replicam a estrutura do relatório real.

In [ ]:
mock_csv = """
DtBase,Autorzc,Cd,FatAnual,IniRelactCli,PorteCli,Tp,TpCtrl,CEP,Contrt,DetCli,DtaProxParcela,DiaAtraso,DtContr,DtVencOp,Indx,PercIndx,IPOC,Mod,NatuOp,OrigemRec,ProvConsttd,TaxEft,VarCamb,VlrContr,QtdParcelas,CartProvMin,Product_id,VlrProxParcela,v20,v40,v110,v120,v130,v140,v150,v160,v165,v205,v210,v220,v230,v240,v245,v250,v255,v310,v320,VlrContBr,Tp_01,TotalCli
2025-01,S,12345678901,50000.00,2021-03-15,3,1,,01310100,100001,,2025-02-15,0,2021-03-15,2026-03-15,1,0.0,ABC123,1501,01,01,500.00,1.2500,0,25000.00,36,C5,8,,0.0,0.0,1000.00,800.00,600.00,400.00,200.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3000.00,1501,3
2025-01,S,98765432100,120000.00,2020-06-01,5,1,,04538133,100002,,2025-02-01,5,2020-06-01,2025-06-01,1,0.0,DEF456,1501,01,01,1200.00,1.8000,0,60000.00,60,C5,9,,0.0,0.0,2000.00,1500.00,1000.00,500.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1500.00,0.0,0.0,0.0,0.0,0.0,6500.00,1501,3
2025-01,S,11223344,500000.00,2019-01-10,6,2,01,80010010,100003,12345678000195,2025-03-01,0,2019-01-10,2027-01-10,2,0.5,GHI789,1503,02,02,5000.00,0.9800,0,250000.00,96,C1,10,,0.0,0.0,5000.00,4000.00,3000.00,2000.00,1000.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15000.00,1503,3
""".strip()

df = pd.read_csv(StringIO(mock_csv))
print(f"DataFrame carregado: {len(df)} linhas, {len(df.columns)} colunas")
df.head()

## 4. Geração do XML CADOC 3040

O CADOC 3040 é o **Documento de Crédito** exigido pelo BACEN. O XML segue uma hierarquia:
```
Doc3040
  └── Cli (por cliente)
        └── Op (por operação)
              ├── Venc (distribuição de vencimentos por vértice)
              ├── Inf  (modalidade/produto)
              └── ContInstFinRes4966 (dados de risco de crédito)
```

In [ ]:
# ── Funções auxiliares ───────────────────────────────────────────────────────

def format_element_with_newlines(elem, level=0):
    """Serializa um elemento XML com indentação legível."""
    indent = "    " * level
    attributes = "".join([f'\n{indent}    {k}="{v}"' for k, v in elem.attrib.items()])
    content = "".join([format_element_with_newlines(child, level + 1) for child in elem])
    if not content.strip():
        return f"{indent}<{elem.tag}{attributes}/>\n"
    return f"{indent}<{elem.tag}{attributes}>\n{content}{indent}</{elem.tag}>\n"


def _attr(row, col):
    """Retorna o valor de uma coluna como string, tratando NaN/None."""
    v = row.get(col, "")
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return ""
    except Exception:
        pass
    return str(v)


def _tp_norm(val):
    """Normaliza o código de tipo de operação para 4 dígitos."""
    if val is None:
        return ""
    s = str(val).strip()
    if not s or s.lower() in ("nan", "none", "null"):
        return ""
    try:
        s = str(int(float(s)))
    except Exception:
        pass
    return s.zfill(4) if s.isdigit() else s


def _num(v):
    """Converte qualquer valor numérico para float, retornando 0.0 em caso de erro."""
    if v is None:
        return 0.0
    try:
        if isinstance(v, float):
            return 0.0 if math.isnan(v) else float(v)
        s = str(v).strip().replace(',', '.')
        return float(s) if s and s.lower() not in ("nan", "none", "null") else 0.0
    except Exception:
        return 0.0


def _valid_attr(row, key):
    """Retorna o valor apenas se for não-nulo e diferente de zero."""
    v = str(_attr(row, key)).strip()
    return v if v and v not in ('0', '0.0', '0.00', '0,0', '0,00') else None


print("Funções auxiliares definidas.")

In [ ]:
# ── Parâmetros do cabeçalho (substituir com dados reais em produção) ─────────
CNPJ_INSTITUICAO  = "00000000"
NOME_RESPONSAVEL  = "Nome do Responsável"
EMAIL_RESPONSAVEL = "responsavel@instituicao.com.br"
TEL_RESPONSAVEL   = "1100000000"

# ── CNPJs das custodiantes/cedentes por produto ──────────────────────────────
IDENT_PRODUTO_10 = "00000000000000"
IDENT_PRODUTO_8  = "00000000000000"
IDENT_PRODUTO_9  = "00000000000000"

# ── Vértices de prazo reconhecidos pelo BACEN ────────────────────────────────
VENC_KEYS = [
    'v20','v40','v110','v120','v130','v140','v150','v160','v165',
    'v205','v210','v220','v230','v240','v245','v250','v255','v310','v320'
]

# ── Geração do XML ───────────────────────────────────────────────────────────
total_cli_value = str(df['TotalCli'].iloc[0]) if 'TotalCli' in df.columns and not df.empty else "0"
date = str(df['DtBase'].iloc[0])

documento = Element('Doc3040', {
    'DtBase': date,
    'CNPJ': CNPJ_INSTITUICAO,
    'TotalCli': total_cli_value,
    'Remessa': "1",
    'Parte': "1",
    'TpArq': "F",
    'NomeResp': NOME_RESPONSAVEL,
    'EmailResp': EMAIL_RESPONSAVEL,
    'TelResp': TEL_RESPONSAVEL,
    'MetodApPE': "S",
    'MetodDifTJE': "N"
})

clientes_map = {}

for _, row in df.iterrows():
    cd  = _attr(row, 'Cd').strip()
    tp  = _attr(row, 'Tp').strip()
    chave   = (cd, tp)
    tp_norm = _tp_norm(row.get('Tp_01'))

    if chave in clientes_map:
        Clientes = clientes_map[chave]
    else:
        cli_attrs = {
            'Autorzc': _attr(row, 'Autorzc'),
            'Cd': cd,
            'FatAnual': _attr(row, 'FatAnual'),
            'IniRelactCli': _attr(row, 'IniRelactCli'),
            'PorteCli': _attr(row, 'PorteCli'),
            'Tp': tp,
        }
        tpctrl = _attr(row, 'TpCtrl').strip()
        if tpctrl and tpctrl not in ('0', '0.0'):
            cli_attrs['TpCtrl'] = tpctrl
        Clientes = SubElement(documento, 'Cli', cli_attrs)
        clientes_map[chave] = Clientes

    soma_carac19 = sum(_num(row.get(k)) for k in ('v240', 'v245', 'v250', 'v255'))
    precisa_carac19 = soma_carac19 > 0

    op_attrs = {}
    if tp == '2':
        op_attrs['DetCli'] = _attr(row, 'DetCli')
    op_attrs.update({
        'Contrt': _attr(row, 'Contrt'),
        'IPOC': _attr(row, 'IPOC'),
        'Mod': _attr(row, 'Mod'),
        'OrigemRec': _attr(row, 'OrigemRec'),
        'Indx': _attr(row, 'Indx'),
        'PercIndx': _attr(row, 'PercIndx'),
        'VarCamb': _attr(row, 'VarCamb'),
        'CEP': _attr(row, 'CEP'),
        'TaxEft': _attr(row, 'TaxEft'),
        'DtContr': _attr(row, 'DtContr'),
        'NatuOp': _attr(row, 'NatuOp'),
        'DtVencOp': _attr(row, 'DtVencOp'),
    })
    dia_atraso = _attr(row, 'DiaAtraso').strip()
    if dia_atraso and dia_atraso not in ('0', '0.0'):
        op_attrs['DiaAtraso'] = dia_atraso
    if tp_norm != '0399':
        op_attrs['ProvConsttd'] = _attr(row, 'ProvConsttd') or '0.00'
    for opt_key in ('DtaProxParcela', 'VlrProxParcela', 'QtdParcelas', 'VlrContr'):
        v = _attr(row, opt_key)
        if v:
            op_attrs[opt_key] = v
    if precisa_carac19:
        op_attrs['CaracEspecial'] = '19'

    Operacao = SubElement(Clientes, 'Op', op_attrs)

    venc_attrs = {k: v for k in VENC_KEYS if (v := _valid_attr(row, k))}

    def _tje(row):
        return float(
            Decimal(str(row.get('TaxEft') or '0').replace(',', '.'))
            .quantize(Decimal('0.0000000'), rounding=ROUND_HALF_UP)
        )

    if tp_norm == '0399':
        SubElement(Operacao, 'Inf', {'Tp': '0399'})

    elif tp_norm in ('1501', '1503'):
        if venc_attrs:
            SubElement(Operacao, 'Venc', venc_attrs)
        product_id = int(row.get('Product_id', 0))
        if tp_norm == '1503':
            SubElement(Operacao, 'Inf', {'Tp': tp_norm, 'Ident': IDENT_PRODUTO_10})
        elif product_id == 8:
            SubElement(Operacao, 'Inf', {'Tp': tp_norm, 'Ident': IDENT_PRODUTO_8})
        elif product_id == 9:
            SubElement(Operacao, 'Inf', {'Tp': tp_norm, 'Ident': IDENT_PRODUTO_9})
        SubElement(Operacao, 'ContInstFinRes4966', {
            'ClasAtFin': "1",
            'VlrContBr': _attr(row, 'VlrContBr'),
            'TJE': f"{_tje(row):.7f}",
            'CartProvMin': _attr(row, 'CartProvMin'),
        })

    else:
        if venc_attrs:
            SubElement(Operacao, 'Venc', venc_attrs)
        SubElement(Operacao, 'ContInstFinRes4966', {
            'ClasAtFin': "1",
            'VlrContBr': _attr(row, 'VlrContBr'),
            'TJE': f"{_tje(row):.7f}",
            'CartProvMin': _attr(row, 'CartProvMin'),
        })

print(f"XML gerado com {len(clientes_map)} clientes.")

## 5. Exportação do Arquivo XML

In [ ]:
import os
from datetime import datetime

OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

hoje = datetime.today().strftime("%d%m%Y")
nome_arquivo = f"{OUTPUT_DIR}/CADOC3040_{CNPJ_INSTITUICAO}_{hoje}.xml"

xml_str = format_element_with_newlines(documento)
with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write(xml_str)

print(f"Arquivo gerado: {nome_arquivo}")
print("\nPrimeiras linhas do XML:")
print("\n".join(xml_str.split("\n")[:20]))